In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
from src.config import *
from src.utils import *
import matplotlib.pyplot as plt

In [ ]:
gt = pd.read_csv(BINARY_GT_TRAIN_CSV)
gt.head()

#### Random sample of images from the dataset.

In [ ]:
sample = random_sample(gt)
images = [load_image(r.filename, r.label) for r in sample.itertuples()]
titles = [f"{r.filename} | {r.label}" for r in sample.itertuples()]
show_grid(images, titles)

#### Exploring some thresholds for obtaining a valid mask.
The valid mask is used to filter out pixels that are too dark or too bright, which can help improve the quality of the images for training.

In [ ]:
r = random_sample(gt, n=1).iloc[0]
img = load_image(r.filename, r.label)

thresholds = [20, 40, 60, 80, 100]
fig, axes = plt.subplots(1, len(thresholds), figsize=(18, 4))
for ax, t in zip(axes, thresholds):
    mask = get_valid_mask(img, threshold=t)
    overlay = img.copy()
    overlay[~mask] = [255, 0, 0]
    ax.imshow(overlay)
    ax.set_title(f"t={t} | valid: {mask.mean():.0%}")
    ax.axis("off")
plt.show()

### Obtain the vlaid mask from images

In [ ]:
sample = random_sample(gt)

images, titles = [], []
for r in sample.itertuples():
    img = load_image(r.filename, r.label)
    mask = get_valid_mask(img, threshold=40) # A threshold of 40 seems to be doing good enough, not under or over segmentation of the dark halos
    overlay = img.copy()
    overlay[~mask] = [255, 0, 0]
    images.append(overlay)
    titles.append(f"{r.filename} | valid: {mask.mean():.0%}")

show_grid(images, titles)

### Crop to the valid region
Here the valid region is everything that is not coloured as red, so halos are excluded from being `valid` pixels (`valid` = ~ `halo`). The crop is done to biggest rectangle that can fit inside the valid region, so that the cropped image does not contain any halos.

In [ ]:
vignette_imgs = []
for r in gt.sample(300).itertuples():
    img = load_image(r.filename, r.label)
    valid = get_valid_mask(img)
    if has_vignette(valid):
        vignette_imgs.append((img, crop_inside_halo(img, valid), r.filename))
    if len(vignette_imgs) == 9:
        break

show_grid([v[1] for v in vignette_imgs],
          [f"{v[2]} | {v[0].shape[1]}x{v[0].shape[0]} -> {v[1].shape[1]}x{v[1].shape[0]}" for v in vignette_imgs])

### Resize all the images to a common size and crop the lesion
This is not straightforward: we must use the shortest side of all the images as the resizing dimension. Additionally, we should maintain the aspect ratio of the images in order to avoid distorting the lesions.

We will explore the distribution of the shortest side of all the images after cropping, and choose a resizing dimension that is not too small (to avoid enlarging too many images) and not too large (to avoid downsampling too many images).

In [ ]:
shorter_sides = []
print('Gathering the shortest side of each cropped image. It might take some minutes...')
for r in gt.itertuples():
    img = load_image(r.filename, r.label)
    valid = get_valid_mask(img)
    if has_vignette(valid):
        img = crop_inside_halo(img, valid)
    shorter_sides.append(min(img.shape[:2]))

gt["shorter_side"] = shorter_sides
print(round(gt["shorter_side"].describe(percentiles=[0.01, 0.05, 0.5])), 2)

In [ ]:
shorter = gt["shorter_side"]
bins = np.linspace(shorter.min(), shorter.max(), 101)   # 100 bins from min to max

plt.figure(figsize=(10, 4))
plt.hist(shorter, bins=bins)
plt.axvline(shorter.min(), color="red", linestyle="--", label=f"min = {shorter.min()}")
plt.axvline(shorter.quantile(0.01), color="orange", linestyle="--", label=f"1% = {int(shorter.quantile(0.01))}")
plt.xlabel("Shorter side after vignette crop (px)")
plt.ylabel("log(Number of images)")
plt.legend()
plt.yscale("log")
plt.show()

In [ ]:
print(f"Below 450:   {(shorter < 450).mean():.2%}")
print(f"Up to 450:   {(shorter <= 450).mean():.2%}")

This last plot tells us that useing the minimum value of the `shorter_side` vector might not be a good idea. Instead, we can choose **450 px as the resizing dimension**, which will only enlarge (this is more critical since it needs interpolation) about 2% of the images, while the rest will be downsampled (which is less critical since it is just removing pixels).

In [ ]:
resizing_dim = 450

sample = random_sample(gt)

images, titles = [], []
for r in sample.itertuples():
    img = load_image(r.filename, r.label)
    original_size = f"{img.shape[1]}x{img.shape[0]}"

    # Step 1: remove vignette
    valid = get_valid_mask(img)
    if has_vignette(valid):
        img = crop_inside_halo(img, valid)

    # Step 2: common size
    img = resize_and_center_crop(img, resizing_dim)

    images.append(img)
    titles.append(f"{r.filename} | {original_size} -> {img.shape[1]}x{img.shape[0]}")

show_grid(images, titles)

The cropping results look fine. We should also take a look at the results of those images that got upscaled during resizing.

In [ ]:
upscaled = gt[gt["shorter_side"] < resizing_dim]
print(upscaled["label"].value_counts())

sample = random_sample(upscaled)

images, titles = [], []
for r in sample.itertuples():
    img = load_image(r.filename, r.label)

    # Firstly remove vignette
    valid = get_valid_mask(img)
    if has_vignette(valid):
        img = crop_inside_halo(img, valid)

    # Secondly resize to common size
    img = resize_and_center_crop(img, resizing_dim)

    factor = resizing_dim / r.shorter_side
    titles.append(f"{r.filename} | {r.shorter_side}px -> {resizing_dim}px (x{factor:.2f})")
    images.append(img)

show_grid(images, titles)

By visual inspection, we can deliver that the upscaling of the images with a side smaller than 450px did not cause any abnormalities at the naked eye. Additionally, since the cropping produced square images, the upscaling did not cause any non isotropic stretches in the lesions of the images, so data integrity is preserved across the dataset.

### Hair removal
Since the objective is to obtain features from the lesions such as color, texture, shape, etc. the hairs in the skin would affect the related metrics if we did not remove them.

This process consists in 3 steps:
1. **Black-hat filter** to highlight thin structures that are darker than their surroundings, such as hairs.
2. **Thresholding** to obtain the mask of the hairs.
3. **Filling** the hair pixels using the surrounding skin intensity.

In [ ]:
"""sample = random_sample(gt, seed=188)

images, titles = [], []
for r in sample.itertuples():
    img = load_image(r.filename, r.label) # Load the image
    valid = get_valid_mask(img) # Obtain the pixels inside the halo
    if has_vignette(valid): # If the obtained pixels do not coincide with the entire image, and therefore it has a halo...
        img = crop_inside_halo(img, valid)  # Obtain the biggest bbox inside the halo
    img = resize_and_center_crop(img, resizing_dim) # And resize and center the image to 450x450 px

    kernel_size = 17
    threshold = 20
    min_length = 40
    max_width = 8
    clean, hair_mask = remove_hair(img, kernel_size, threshold, min_length, max_width)

    overlay = img.copy()
    overlay[hair_mask > 0] = [255, 0, 0]
    images.append(overlay)
    titles.append(f"{r.filename} | hair: {(hair_mask > 0).mean():.1%}")

show_grid(images, titles)"""

In [ ]:
images, titles = [], []
sample = random_sample(gt)
for r in sample.itertuples():
    img = load_image(r.filename, r.label)
    valid = get_valid_mask(img)
    if has_vignette(valid):
        img = crop_inside_halo(img, valid)
    img = resize_and_center_crop(img, resizing_dim)

    kernel_size = 17
    threshold = 15
    min_length = 40
    max_width = 10

    candidates = detect_hair_candidates(img, kernel_size, threshold)
    hair_mask = keep_hair_components(candidates, min_length, max_width)

    overlay = img.copy()
    overlay[(candidates > 0) & (hair_mask == 0)] = [255, 255, 0]   # yellow = rejected
    overlay[hair_mask > 0] = [255, 0, 0]                           # red = kept as hair
    images.append(overlay)
    titles.append(f"{r.filename} | hair: {(hair_mask > 0).mean():.1%}")

show_grid(images, titles)

In [ ]:
images = []
for r in sample.itertuples():
    img = load_image(r.filename, r.label)
    valid = get_valid_mask(img)
    if has_vignette(valid):
        img = crop_inside_halo(img, valid)
    img = resize_and_center_crop(img, resizing_dim)
    clean, _ = remove_hair(img, kernel_size, threshold, min_length)
    images.append(clean)

show_grid(images, [r.filename for r in sample.itertuples()])

### Color Consistency through "Shades of Gray"
This final processing step is to make illumination even across the entire dataset.

This step consists in estimating the light color, normalise it and then use this normalised value to correct each RGB channel.

In [ ]:
before = []
for r in sample.itertuples():
    img = load_image(r.filename, r.label)
    valid = get_valid_mask(img)
    if has_vignette(valid):
        img = crop_inside_halo(img, valid)
    img = resize_and_center_crop(img, resizing_dim)
    img, _ = remove_hair(img, kernel_size, threshold, min_length, max_width)
    before.append(img)

show_grid(before, [r.filename for r in sample.itertuples()])

In [ ]:
after = [shades_of_gray(img) for img in before]
show_grid(after, [r.filename for r in sample.itertuples()])

## Execute the entire pipeline and create a new processed dataset

In [ ]:
params = dict(
    resizing_dim=450,
    kernel_size=17,
    threshold=10,
    min_length=40,
    max_width=10,
)

In [ ]:
sample = random_sample(gt)
images = [preprocess_image(load_image(r.filename, r.label), **params) for r in sample.itertuples()]
show_grid(images, [r.filename for r in sample.itertuples()])

In [ ]:
gt_val = build_ground_truth(BINARY_VAL_DIR)
gt_val.to_csv(BINARY_GT_VAL_CSV, index=False)
print(gt_val["label"].value_counts())

In [ ]:
preprocess_dataset(gt, BINARY_TRAIN_DIR, BINARY_PREPROC_TRAIN_DIR, **params)
preprocess_dataset(gt_val, BINARY_VAL_DIR, BINARY_PREPROC_VAL_DIR, **params)